# Prompt Eval Results Viewer

Load and compare `EvaluationReport` JSON files produced by `prompt_eval`.

Usage:
- Results JSONs live in `data/` (e.g. `data/baseline_abstract.json`)
- Run all cells to load and display results
- Use the comparison section to diff two runs side-by-side

In [1]:
from pathlib import Path
import json
import pandas as pd
from llm_metadata.groundtruth_eval import EvaluationReport

# Locate project root robustly (works whether Jupyter started from root or notebooks/)
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"

# List available result JSON files
result_files = sorted(DATA_DIR.glob("*.json"))
print(f"Found {len(result_files)} result files in {DATA_DIR}:")
for f in result_files:
    print(f"  {f.name}")

Found 2 result files in c:\Users\beav3503\dev\llm_metadata\data:
  baseline_abstract.json
  baseline_pdf.json


## Load a Single Run

In [2]:
# Load a single run — edit path as needed
RUN_PATH = DATA_DIR / "baseline_abstract.json"

if RUN_PATH.exists():
    report = EvaluationReport.load(RUN_PATH)
    
    # Show run metadata
    with open(RUN_PATH) as f:
        meta = json.load(f)
    
    print("Run metadata:")
    for k, v in meta.items():
        if k not in ("field_metrics", "field_results", "config"):
            print(f"  {k}: {v}")
else:
    print(f"File not found: {RUN_PATH}")
    print("Run prompt_eval first to generate results:")
    print("  uv run python -m llm_metadata.prompt_eval --subset data/dev_subset.csv --output data/baseline_abstract.json")
    report = None

Run metadata:
  prompt_module: prompts.abstract
  model: gpt-5-mini
  cost_usd: 0.1266
  timestamp: 2026-02-19T18:47:37.084975+00:00


## Per-Field Metrics Table

In [3]:
if report is not None:
    metrics_df = report.metrics_to_pandas()
    metrics_df = metrics_df.sort_values("f1", ascending=False)
    
    # Format for display
    display_df = metrics_df[["field", "n", "tp", "fp", "fn", "precision", "recall", "f1"]].copy()
    for col in ["precision", "recall", "f1"]:
        display_df[col] = display_df[col].map(lambda x: f"{x:.3f}" if x is not None else "-")
    
    display(display_df.reset_index(drop=True))

,field,n,tp,fp,fn,precision,recall,f1
0,temp_range_i,17,5,0,3,1.000,0.625,0.769
1,temp_range_f,17,5,0,3,1.000,0.625,0.769
2,multispecies,17,12,3,5,0.800,0.706,0.750
3,threatened_species,17,2,0,6,1.000,0.250,0.400
4,species,17,14,54,18,0.206,0.438,0.280
5,time_series,17,1,6,0,0.143,1.000,0.250
6,data_type,17,7,30,13,0.189,0.350,0.246
7,new_species_region,17,1,1,8,0.500,0.111,0.182
8,geospatial_info_dataset,17,4,34,4,0.105,0.500,0.174
9,bias_north_south,17,0,0,17,nan,0.000,nan


## Mismatch Explorer

In [4]:
if report is not None:
    FIELD = "species"  # Change to inspect different fields
    
    errors = report.errors_for_field(FIELD)
    print(f"Mismatches for '{FIELD}': {len(errors)} / {report.field_metrics[FIELD].n if FIELD in report.field_metrics else '?'}")
    
    for r in errors[:10]:  # Show first 10 mismatches
        print(f"\n  record_id: {r.record_id}")
        print(f"  true:  {r.true_value}")
        print(f"  pred:  {r.pred_value}")
        print(f"  tp={r.tp}, fp={r.fp}, fn={r.fn}")

Mismatches for 'species': 11 / 17

  record_id: 175
  true:  ['sapin baumier', 'epinette blanche', 'bouleau jaune', 'bouleau a papier', 'Crable a epis', 'cerisier de Pennsylvanie']
  pred:  ['balsam fir (Abies balsamea (L.) Mill.)', 'yellow birch (Betula alleghaniensis Britton)', 'mountain maple (Acer spicatum Lam.)', 'pin cherry (Prunus pensylvanica (L.))']
  tp=0, fp=4, fn=6

  record_id: 208
  true:  ['c.132 species of benthic community']
  pred:  None
  tp=0, fp=0, fn=1

  record_id: 225
  true:  ['Benthic intertidal community (c.20 algae', 'c.15 molluscs', 'c.15 annelid species', 'and others)']
  pred:  ['Fucus distichus edentatus', 'F. vesiculosus', 'Mytilus spp.', 'macroalgae', 'mussels', 'habitat-forming species (HFS)']
  tp=0, fp=6, fn=4

  record_id: 258
  true:  ['16 damselfly species', 'water mite']
  pred:  ['Coenagrionidae (damselflies)', 'Arrenuridae (water mites)', '16 damselfly species']
  tp=2, fp=1, fn=0

  record_id: 271
  true:  ['11 mite species']
  pred:  ['Ambly

## Side-by-Side Comparison of Two Runs

In [6]:
RUN_A = DATA_DIR / "baseline_abstract.json"
RUN_B = DATA_DIR / "baseline_pdf.json"  # second run to compare

if RUN_A.exists() and RUN_B.exists():
    report_a = EvaluationReport.load(RUN_A)
    report_b = EvaluationReport.load(RUN_B)
    
    metrics_a = report_a.metrics_to_pandas().set_index("field")
    metrics_b = report_b.metrics_to_pandas().set_index("field")
    
    common_fields = sorted(set(metrics_a.index) & set(metrics_b.index))
    
    rows = []
    for field in common_fields:
        f1_a = metrics_a.loc[field, "f1"]
        f1_b = metrics_b.loc[field, "f1"]
        delta = (f1_b - f1_a) if (f1_a is not None and f1_b is not None) else None
        rows.append({
            "field": field,
            "f1_A": f"{f1_a:.3f}" if f1_a is not None else "-",
            "f1_B": f"{f1_b:.3f}" if f1_b is not None else "-",
            "delta": f"{delta:+.3f}" if delta is not None else "-",
        })
    
    comparison_df = pd.DataFrame(rows)
    display(comparison_df)
else:
    missing = [str(p) for p in [RUN_A, RUN_B] if not p.exists()]
    print(f"Missing result files: {missing}")
    print("Run prompt_eval to generate results, then set RUN_A / RUN_B above.")

,field,f1_A,f1_B,delta
0,bias_north_south,nan,nan,+nan
1,data_type,0.246,0.280,+0.034
2,geospatial_info_dataset,0.174,0.154,-0.020
3,multispecies,0.750,0.538,-0.212
4,new_species_region,0.182,0.308,+0.126
5,new_species_science,nan,0.167,+nan
6,spatial_range_km2,nan,0.133,+nan
7,species,0.280,0.093,-0.187
8,temp_range_f,0.769,0.250,-0.519
9,temp_range_i,0.769,0.250,-0.519
